# FIFA ranking scraper — 2026

This notebook downloads the FIFA men's ranking for 2026 and saves it as a CSV file used in the final dataset.

In [1]:
# Install the packages needed for scraping and parsing the FIFA page.
%pip install -q pandas beautifulsoup4 playwright
!python -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## Setup

Imports libraries, sets the FIFA ranking URL and defines helper functions for extracting rankings.

In [2]:
# Import libraries and define settings for this ranking year.
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


URL = "https://inside.fifa.com/fifa-world-ranking/men"
OUTPUT_FILE = "fifa_rankings_2026.csv"
OUTPUT_FILE_WITH_POINTS = "fifa_rankings_2026_points.csv"
KEEP_POINTS = False
MIN_EXPECTED_ROWS = 200

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

API_URL_HINTS = [
    "ranking-overview",
    "fifa-world-ranking",
    "rankings",
    "ranking",
    "rank",
]


# Scraper class that tries API payloads first and then falls back to HTML parsing.
class FifaRankingScraper:
    def __init__(self, url=URL, min_expected_rows=MIN_EXPECTED_ROWS):
        self.url = url
        self.min_expected_rows = min_expected_rows
        self.api_payloads = []
        self.seen_urls = []
        self.html = ""

    @staticmethod
    def clean_text(text):
        # Standardizes whitespace in scraped text.
        if text is None:
            return ""
        return re.sub(r"\s+", " ", str(text)).strip()

    @classmethod
    def to_float(cls, value):
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return float(value)

        text = cls.clean_text(value).replace(",", "")
        match = re.search(r"-?\d+(?:\.\d+)?", text)
        return float(match.group(0)) if match else None

    @classmethod
    def to_int(cls, value):
        if value is None:
            return None
        if isinstance(value, int):
            return value

        text = cls.clean_text(value)
        match = re.search(r"\d+", text)
        return int(match.group(0)) if match else None

    @classmethod
    def normalize_team_name(cls, name):
        name = cls.clean_text(name)
        name = re.sub(r"^[A-Z]{2,3}\s+", "", name).strip()

        bad_names = {
            "", "team", "rank", "points", "latest results",
            "last result", "more", "ft",
        }
        if name.lower() in bad_names:
            return ""

        return name

    @classmethod
    def finalize_df(cls, df):
        # Cleans ranking rows and keeps one valid team per rank.
        if df is None or df.empty:
            return pd.DataFrame(columns=["rank", "team", "points"])

        df = df.copy()
        if "rank" not in df.columns or "team" not in df.columns:
            return pd.DataFrame(columns=["rank", "team", "points"])

        df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
        df = df.dropna(subset=["rank"])
        df["rank"] = df["rank"].astype(int)

        df["team"] = df["team"].map(cls.normalize_team_name)
        df = df[df["team"].astype(bool)]

        if "points" not in df.columns:
            df["points"] = None
        else:
            df["points"] = pd.to_numeric(df["points"], errors="coerce")

        df = df.drop_duplicates(subset=["rank", "team"])
        df = df.sort_values(["rank", "team"]).reset_index(drop=True)
        df["team_len"] = df["team"].str.len()

        df = (
            df.sort_values(["rank", "team_len"])
              .drop_duplicates(subset=["rank"], keep="first")
              .drop(columns=["team_len"])
              .sort_values("rank")
              .reset_index(drop=True)
        )

        return df[["rank", "team", "points"]]

    @classmethod
    def extract_rows_from_json(cls, obj):
        # Extracts ranking rows from JSON responses captured in the browser.
        rows = []

        def candidate_name(d):
            team_obj = d.get("team") if isinstance(d.get("team"), dict) else {}
            country_obj = d.get("country") if isinstance(d.get("country"), dict) else {}
            association_obj = d.get("association") if isinstance(d.get("association"), dict) else {}

            candidates = [
                team_obj.get("name"),
                team_obj.get("shortName"),
                team_obj.get("fullName"),
                team_obj.get("countryName"),
                country_obj.get("name"),
                country_obj.get("countryName"),
                association_obj.get("name"),
                association_obj.get("countryName"),
                d.get("teamName"),
                d.get("name"),
                d.get("country"),
                d.get("countryName"),
                d.get("associationName"),
            ]

            for candidate in candidates:
                candidate = cls.normalize_team_name(candidate)
                if candidate and not candidate.isdigit():
                    return candidate

            return None

        def candidate_rank(d):
            keys = [
                "rank", "ranking", "position", "rankPosition",
                "rankingPosition", "currentRank", "currentRanking",
            ]
            for key in keys:
                value = cls.to_int(d.get(key))
                if value is not None and 1 <= value <= 300:
                    return value
            return None

        def candidate_points(d):
            keys = [
                "totalPoints", "points", "score", "rankPoints",
                "rankingPoints", "currentPoints", "previousPoints",
            ]
            for key in keys:
                value = cls.to_float(d.get(key))
                if value is not None:
                    return value
            return None

        def walk(value):
            if isinstance(value, dict):
                rank = candidate_rank(value)
                name = candidate_name(value)
                points = candidate_points(value)

                if rank is not None and name:
                    rows.append({"rank": rank, "team": name, "points": points})

                for nested in value.values():
                    walk(nested)

            elif isinstance(value, list):
                for item in value:
                    walk(item)

        walk(obj)
        return cls.finalize_df(pd.DataFrame(rows))

    @classmethod
    def scrape_from_html(cls, html):
        # Fallback parser when the ranking is available only in the page HTML.
        soup = BeautifulSoup(html, "html.parser")
        rows = []

        for tr in soup.select("table tbody tr, tr"):
            cells = tr.find_all(["td", "th"])
            if len(cells) < 2:
                continue

            rank = cls.to_int(cells[0].get_text(" ", strip=True))
            if rank is None or not (1 <= rank <= 300):
                continue

            team = None
            for img in tr.find_all("img"):
                alt = cls.normalize_team_name(img.get("alt"))
                if alt:
                    team = alt
                    break

            if not team:
                team = cls.normalize_team_name(cells[1].get_text(" ", strip=True))

            points = None
            for cell in reversed(cells):
                value = cls.to_float(cell.get_text(" ", strip=True))
                if value is not None and value > 100:
                    points = value
                    break

            if rank and team:
                rows.append({"rank": rank, "team": team, "points": points})

        df = cls.finalize_df(pd.DataFrame(rows))
        if len(df) > 10:
            return df

        text = soup.get_text("\n")
        lines = [cls.clean_text(line) for line in text.splitlines()]
        lines = [line for line in lines if line]

        rows = []
        for i, line in enumerate(lines):
            if not re.fullmatch(r"\d{1,3}", line):
                continue

            rank = int(line)
            if not (1 <= rank <= 300):
                continue

            for j in range(i + 1, min(i + 8, len(lines))):
                candidate = cls.normalize_team_name(lines[j])
                if (
                    candidate
                    and not re.fullmatch(r"\d+(\.\d+)?", candidate)
                    and candidate.lower() != "ft"
                    and len(candidate) > 1
                ):
                    rows.append({"rank": rank, "team": candidate, "points": None})
                    break

        return cls.finalize_df(pd.DataFrame(rows))

    async def accept_cookies(self, page):
        selectors = [
            "button:has-text('Accept')",
            "button:has-text('Accept all')",
            "button:has-text('I Accept')",
            "button:has-text('Agree')",
            "button:has-text('Allow all')",
            "text=Accept All",
        ]

        for selector in selectors:
            try:
                locator = page.locator(selector)
                if await locator.count() > 0:
                    await locator.first.click(timeout=3000)
                    await page.wait_for_timeout(1000)
                    return True
            except Exception:
                pass

        return False

    async def click_show_full_rankings(self, page, max_scrolls=18):
        selectors = [
            "button:has-text('Show full rankings')",
            "text=Show full rankings",
            "button:has-text('Show all rankings')",
            "text=Show all rankings",
            "button:has-text('Full rankings')",
            "button:has-text('All rankings')",
        ]

        for _ in range(max_scrolls):
            for selector in selectors:
                try:
                    locator = page.locator(selector)
                    if await locator.count() == 0:
                        continue

                    button = locator.first
                    try:
                        await button.scroll_into_view_if_needed(timeout=5000)
                    except Exception:
                        pass

                    await page.wait_for_timeout(500)

                    try:
                        await button.click(timeout=5000)
                    except Exception:
                        await button.click(timeout=5000, force=True)

                    print("Clicked: Show full rankings / Show all rankings")
                    await page.wait_for_timeout(6000)
                    return True

                except Exception:
                    pass

            await page.mouse.wheel(0, 1400)
            await page.wait_for_timeout(900)

        print("The Show full rankings / Show all rankings button could not be found.")
        return False

    async def scroll_to_load_all(self, page, rounds=35):
        last_height = 0
        stable_rounds = 0

        for _ in range(rounds):
            height = await page.evaluate("document.body.scrollHeight")
            await page.mouse.wheel(0, 1600)
            await page.wait_for_timeout(700)
            new_height = await page.evaluate("document.body.scrollHeight")

            stable_rounds = stable_rounds + 1 if new_height == last_height else 0
            last_height = new_height

            if stable_rounds >= 5:
                break

    async def extract_from_dom_with_js(self, page):
        body_text = await page.evaluate("() => document.body.innerText || ''")
        escaped = (
            body_text.replace("&", "&amp;")
                     .replace("<", "&lt;")
                     .replace(">", "&gt;")
                     .replace("\n", "<br>")
        )
        return self.scrape_from_html("<html><body>" + escaped + "</body></html>")

    async def _handle_response(self, response):
        url = response.url
        lowered = url.lower()

        if not any(hint in lowered for hint in API_URL_HINTS):
            return

        try:
            content_type = response.headers.get("content-type", "")
            if "json" not in content_type.lower():
                return

            payload = await response.json()
            self.api_payloads.append(payload)
            self.seen_urls.append(url)
        except Exception:
            pass

    async def scrape(self, headless=True, debug=True, timeout_ms=60000):
        self.api_payloads = []
        self.seen_urls = []
        self.html = ""
        dom_df = pd.DataFrame(columns=["rank", "team", "points"])

        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=headless,
                args=[
                    "--disable-blink-features=AutomationControlled",
                    "--no-sandbox",
                    "--disable-dev-shm-usage",
                ],
            )
            context = await browser.new_context(
                viewport={"width": 1440, "height": 1000},
                user_agent=USER_AGENT,
                extra_http_headers={
                    "Accept-Language": "en-US,en;q=0.9",
                    "Referer": "https://www.fifa.com/",
                },
            )
            page = await context.new_page()
            page.on("response", self._handle_response)

            print(f"Loading: {self.url}")

            try:
                await page.goto(self.url, wait_until="domcontentloaded", timeout=timeout_ms)
            except PlaywrightTimeoutError as exc:
                print(f"page.goto timeout, continuing: {exc}")
            except Exception as exc:
                print(f"page.goto issue, continuing: {exc}")

            await self.accept_cookies(page)
            await page.wait_for_timeout(5000)

            clicked = await self.click_show_full_rankings(page)
            await self.scroll_to_load_all(page)

            await page.wait_for_timeout(3000)
            self.html = await page.content()
            dom_df = await self.extract_from_dom_with_js(page)

            if debug:
                Path("fifa_ranking_debug.html").write_text(self.html, encoding="utf-8")
                try:
                    await page.screenshot(path="fifa_ranking_debug.png", full_page=True)
                except Exception:
                    pass

                print(f"Show full rankings / Show all rankings clicked: {clicked}")
                if self.seen_urls:
                    print("Captured possible API URLs:")
                    for url in self.seen_urls[:20]:
                        print("-", url)
                else:
                    print("No suitable JSON API URL was captured.")

            await browser.close()

        candidates = []
        for payload in self.api_payloads:
            df = self.extract_rows_from_json(payload)
            if not df.empty:
                candidates.append(("json", df))

        html_df = self.scrape_from_html(self.html)
        if not html_df.empty:
            candidates.append(("html", html_df))

        if not dom_df.empty:
            candidates.append(("dom", dom_df))

        if not candidates:
            return pd.DataFrame(columns=["rank", "team", "points"])

        source, best_df = max(candidates, key=lambda item: len(item[1]))
        best_df = self.finalize_df(best_df)

        print(f"Data source used: {source}")
        print(f"Number of retrieved rows: {len(best_df)}")

        if len(best_df) <= 10:
            print("WARNING: The scraper still retrieved only 10 or fewer teams.")
            print("Try running the scraper with headless=False:")
            print("df_raw = await scrape_fifa_ranking(headless=False, debug=True)")
        elif len(best_df) < self.min_expected_rows:
            print(f"WARNING: Retrieved fewer than the expected {self.min_expected_rows} rows.")

        return best_df


# Opens the FIFA page, captures ranking data and returns a cleaned dataframe.
async def scrape_fifa_ranking(headless=True, debug=True, timeout_ms=60000):
    scraper = FifaRankingScraper()
    return await scraper.scrape(headless=headless, debug=debug, timeout_ms=timeout_ms)


# Saves the ranking file used later in the final dataset.
def save_ranking(df_raw, keep_points=KEEP_POINTS):
    df_raw = FifaRankingScraper.finalize_df(df_raw)

    if keep_points:
        df = df_raw[["rank", "team", "points"]].copy()
        df.to_csv(OUTPUT_FILE_WITH_POINTS, index=False, encoding="utf-8-sig")
        print(f"Saved: {OUTPUT_FILE_WITH_POINTS}")
    else:
        df = df_raw[["rank", "team"]].copy()
        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print(f"Saved: {OUTPUT_FILE}")

    return df

## Run scraper

Runs the scraper, saves the ranking CSV and shows the first and last rows for control.

In [3]:
# Download and save the ranking table for this year.
df_raw = await scrape_fifa_ranking(headless=True, debug=True)

if df_raw.empty:
    print("Could not retrieve data.")
    print("Try running: df_raw = await scrape_fifa_ranking(headless=False, debug=True)")
else:
    df = save_ranking(df_raw, keep_points=KEEP_POINTS)
    display(df.head(20))
    display(df.tail(20))
    print(f"Number of rows: {len(df)}")

Loading: https://inside.fifa.com/fifa-world-ranking/men
Clicked: Show full rankings / Show all rankings
Show full rankings / Show all rankings clicked: True
Captured possible API URLs:
- https://inside.fifa.com/api/live-world-ranking/get-international-ranking-window?locale=en&date=2026-06-11&category=men
- https://api.fifa.com/api/v3/fifarankings/rankings/live?gender=1&sportType=0&language=en
- https://inside.fifa.com/api/live-world-ranking/get-match-window-matches?locale=en&gender=1&rankingType=0
- https://api.fifa.com/api/v3/fifarankings/rankingMatches/live?gender=1&sportType=0&language=en
Data source used: html
Number of retrieved rows: 211
Saved: fifa_rankings_2026.csv


,rank,team
0,1,Argentina
1,2,Spain
2,3,France
3,4,England
4,5,Portugal
5,6,Brazil
6,7,Morocco
7,8,Netherlands
8,9,Belgium
9,10,Germany


,rank,team
191,192,Bhutan
192,193,Macau
193,194,Brunei
194,195,São Tomé and Príncipe
195,196,Djibouti
196,197,Cayman Islands
197,198,Pakistan
198,199,Somalia
199,200,Tonga
200,201,Timor-Leste


Number of rows: 211


## Check saved CSV

Reads the saved file back into Python to confirm that the export worked.

In [4]:
# Load the saved ranking file as a final check.
df_check = pd.read_csv(OUTPUT_FILE)
df_check.head()

,rank,team
0,1,Argentina
1,2,Spain
2,3,France
3,4,England
4,5,Portugal
